# Character Inventory Agent 테스트 (Production Level)

캐릭터 인벤토리/아이템 추출 에이전트 테스트 노트북

## 역할: "Item Manager" (아이템 관리자)
- **equipped_items**: 장착 중인 아이템 (무기, 방어구)
- **bag_items**: 소지 중인 아이템
- **quest_items**: 퀘스트 아이템
- **currency**: 골드/화폐

> **⚠️ 핵심 검증**: 아이템 언급이 없는 텍스트에서 `has_inventory_data=false`여야 함

In [1]:
import sys, os, json, asyncio
project_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if project_root not in sys.path: sys.path.insert(0, project_root)
from dotenv import load_dotenv
load_dotenv(os.path.join(project_root, '.env'))
print(f"Project root: {project_root}")

Project root: c:\jungle\weapon\sto-link-AI-backend


In [2]:
# 아이템이 명시된 스토리
SAMPLE_STORY_WITH_ITEMS = """아린은 손에 쥔 은빛 검을 꼭 움켜쥐었다. 그녀의 허리에는 가죽 벨트가 있었고, 작은 치유 물약 2개가 달려 있었다.

카엘은 검은 갑옷을 입고 있었다. 그의 손에는 어둠의 대검이 들려 있었고, 허리춤에는 500 골드가 든 주머니가 있었다."""

# 아이템 언급이 없는 스토리
SAMPLE_STORY_NO_ITEMS = """아린은 어두운 숲 한가운데 서 있었다. 긴 검은 머리카락을 바람에 휘날리며 경계심 가득한 눈으로 주위를 살폈다.

그림자 속에서 카엘이 나타났다. 전직 기사는 차가운 눈빛으로 그녀를 바라보았다."""

def create_base_state(story):
    return {"content": story, "completed_agents": [], "errors": [], "messages": []}

def run_async(coro):
    try:
        loop = asyncio.get_event_loop()
        if loop.is_running():
            import nest_asyncio; nest_asyncio.apply()
            return loop.run_until_complete(coro)
        return asyncio.run(coro)
    except: return asyncio.run(coro)

## 1. Inventory Agent 실행 (아이템 있는 경우)

In [3]:
from app.agents.extraction.character.inventory import inventory_extraction_node

async def test_inventory_with_items():
    print("📦 Inventory Agent 테스트 (아이템 있음)...")
    return await inventory_extraction_node(create_base_state(SAMPLE_STORY_WITH_ITEMS))

result_with_items = run_async(test_inventory_with_items())

if result_with_items.get('errors'):
    print(f"\n❌ 에러 발생:")
    for err in result_with_items.get('errors', []):
        print(f"   {err}")
else:
    inventory_data = result_with_items.get('char_inventory', {})
    print(f"\n✅ 추출 완료:")
    print(f"   - 캐릭터 수: {len(inventory_data)}개")
    print(f"   - 이름: {list(inventory_data.keys())}")

📦 Inventory Agent 테스트 (아이템 있음)...

✅ 추출 완료:
   - 캐릭터 수: 2개
   - 이름: ['Arin', 'Kael']


## 2. Human-Readable 출력

In [4]:
inventory_data = result_with_items.get('char_inventory', {})

print("="*70)
print("📦 Inventory Data (캐릭터별 아이템)")
print("="*70)

for name, data in inventory_data.items():
    print(f"\n🧑 {name}")
    
    # Currency
    if data.get('currency'):
        print(f"   💰 화폐: {data['currency']} Gold")
    
    # Equipped Items
    equipped = data.get('equipped_items', [])
    if equipped:
        print(f"   ⚔️ 장착 중:")
        for item in equipped:
            slot = item.get('slot', '')
            slot_str = f"[{slot}]" if slot else ""
            print(f"      - {item['name']} ({item.get('item_type', 'MISC')}) {slot_str}")
    
    # Bag Items
    bag = data.get('bag_items', [])
    if bag:
        print(f"   🎒 소지품:")
        for item in bag:
            qty = item.get('quantity', 1)
            qty_str = f"x{qty}" if qty > 1 else ""
            print(f"      - {item['name']} {qty_str}")
    
    # Quest Items
    quest = data.get('quest_items', [])
    if quest:
        print(f"   🔑 퀘스트 아이템:")
        for item in quest:
            print(f"      - {item}")
    
    if not any([equipped, bag, quest, data.get('currency')]):
        print("   (아이템 없음)")

📦 Inventory Data (캐릭터별 아이템)

🧑 Arin
   ⚔️ 장착 중:
      - Silver Sword (WEAPON) [MAIN_HAND]
   🎒 소지품:
      - Silver Sword 
      - Healing Potion x2

🧑 Kael
   💰 화폐: 500 Gold
   ⚔️ 장착 중:
      - Dark Greatsword (WEAPON) [MAIN_HAND]
   🎒 소지품:
      - Dark Greatsword 
      - Gold Pouch 


## 3. Inventory Agent 실행 (아이템 없는 경우)

In [5]:
async def test_inventory_no_items():
    print("📦 Inventory Agent 테스트 (아이템 없음)...")
    return await inventory_extraction_node(create_base_state(SAMPLE_STORY_NO_ITEMS))

result_no_items = run_async(test_inventory_no_items())
inventory_empty = result_no_items.get('char_inventory', {})

print(f"\n✅ 추출 완료:")
print(f"   - 캐릭터 수: {len(inventory_empty)}개")

# Check for hallucination
has_hallucination = False
for name, data in inventory_empty.items():
    if data.get('equipped_items') or data.get('bag_items') or data.get('currency'):
        has_hallucination = True
        print(f"   ⚠️ {name}: 아이템 환각(Hallucination)됨")

if not has_hallucination:
    print("   ✅ 환각 없음 - 정상")

📦 Inventory Agent 테스트 (아이템 없음)...

✅ 추출 완료:
   - 캐릭터 수: 2개
   ✅ 환각 없음 - 정상


## 4. Full JSON 출력

In [6]:
print("="*70)
print("📄 Full JSON Output (아이템 있는 경우)")
print("="*70)
inventory_data = result_with_items.get('char_inventory', {})
if inventory_data:
    print(json.dumps(inventory_data, ensure_ascii=False, indent=2))
else:
    print("{}")

📄 Full JSON Output (아이템 있는 경우)
{
  "Arin": {
    "name": "Arin",
    "equipped_items": [
      {
        "item_id": null,
        "name": "Silver Sword",
        "item_type": "WEAPON",
        "quantity": 1,
        "rarity": "COMMON",
        "estimated_value": 50,
        "equipped": true,
        "slot": "MAIN_HAND",
        "stats": {
          "attack_bonus": null,
          "defense_bonus": null,
          "hp_bonus": null,
          "mp_bonus": null,
          "special_effect": null
        },
        "description": null
      }
    ],
    "bag_items": [
      {
        "item_id": null,
        "name": "Silver Sword",
        "item_type": "WEAPON",
        "quantity": 1,
        "rarity": "COMMON",
        "estimated_value": 50,
        "equipped": true,
        "slot": "MAIN_HAND",
        "stats": {
          "attack_bonus": null,
          "defense_bonus": null,
          "hp_bonus": null,
          "mp_bonus": null,
          "special_effect": null
        },
        "descri

## 5. Production 체크리스트

In [7]:
print("="*70)
print("✅ Production 체크리스트")
print("="*70)

inventory_data = result_with_items.get('char_inventory', {})
checks = []

# 1. 캐릭터 존재
if len(inventory_data) >= 2:
    checks.append(("✅", f"{len(inventory_data)} characters extracted"))
else:
    checks.append(("⚠️", f"Only {len(inventory_data)} characters"))

if inventory_data:
    # 2. Equipped items
    has_equipped = any(data.get('equipped_items') for data in inventory_data.values())
    if has_equipped:
        checks.append(("✅", "Equipped items extracted"))
    else:
        checks.append(("❌", "No equipped items (expected for item text)"))
    
    # 3. Item types
    item_types = set()
    for data in inventory_data.values():
        for item in data.get('equipped_items', []):
            item_types.add(item.get('item_type'))
        for item in data.get('bag_items', []):
            item_types.add(item.get('item_type'))
    if len(item_types) >= 2:
        checks.append(("✅", f"Multiple item types: {item_types}"))
    else:
        checks.append(("⚠️", f"Only item types: {item_types}"))
    
    # 4. Currency
    has_currency = any(data.get('currency') for data in inventory_data.values())
    if has_currency:
        checks.append(("✅", "Currency extracted"))
    else:
        checks.append(("⚠️", "No currency extracted"))

# 5. 환각 검사
inventory_empty = result_no_items.get('char_inventory', {})
no_hallucination = not any(
    data.get('equipped_items') or data.get('bag_items') or data.get('currency')
    for data in inventory_empty.values()
)
if no_hallucination:
    checks.append(("✅", "No hallucination on non-item text"))
else:
    checks.append(("❌", "Hallucination detected"))

print()
for status, msg in checks:
    print(f"{status} {msg}")

print("\n" + "=" * 70)
passed = sum(1 for s, _ in checks if s == "✅")
print(f"결과: {passed}/{len(checks)} checks passed")

✅ Production 체크리스트

✅ 2 characters extracted
✅ Equipped items extracted
✅ Multiple item types: {'WEAPON', 'MISC', 'CONSUMABLE'}
✅ Currency extracted
✅ No hallucination on non-item text

결과: 5/5 checks passed


## 6. 디버그 정보

In [8]:
print("="*70)
print("🔍 디버그 정보")
print("="*70)
print(f"\n[아이템 있는 텍스트]")
print(f"Result keys: {result_with_items.keys()}")
print(f"Errors: {result_with_items.get('errors', [])}")
print(f"Completed agents: {result_with_items.get('completed_agents', [])}")

print(f"\n[아이템 없는 텍스트]")
print(f"Result keys: {result_no_items.keys()}")
print(f"Errors: {result_no_items.get('errors', [])}")

🔍 디버그 정보

[아이템 있는 텍스트]
Result keys: dict_keys(['char_inventory', 'completed_agents', 'messages'])
Errors: []
Completed agents: ['inventory']

[아이템 없는 텍스트]
Result keys: dict_keys(['char_inventory', 'completed_agents', 'messages'])
Errors: []
